# ODIR-5K Vision Training Pipeline (EfficientNet-B4) on Google Colab

This notebook is designed by ClinicaEye-NLP. It orchestrates the extraction of the ODIR dataset from Google Drive for faster I/O speed, preprocesses the fundus images using **Ben Graham's Technique**, handles heavy medical class imbalances, trains via EfficientNet-B4 utilizing mixed precision, and finally checkpoints the best weights automatically to Google Drive.

In [ ]:
!pip install scikit-multilearn tqdm opencv-python-headless torchvision matplotlib pandas scikit-learn

### 1. Drive Orchestration & Unzipping
We mount the drive and extract the zip archive into the fast `/content/data_odir` local disk storage to avoid Google Drive slow network bottlenecks during preprocessing and training.

In [ ]:
from google.colab import drive
import os
import zipfile

drive.mount('/content/drive')

archive_path = '/content/drive/MyDrive/Okul/ClicinicaEye-NLP/Data/archive.zip'
extract_path = '/content/data_odir'

if not os.path.exists(extract_path):
    os.makedirs(extract_path, exist_ok=True)
    print(f"Extracting {archive_path} to {extract_path}...")
    with zipfile.ZipFile(archive_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete.")
else:
    print(f"{extract_path} already extracted and ready.")

### 2. Offline Preprocessing (Ben Graham's Method)
Tight circular crop around the fundus and localized Gaussian blur color subtraction to remove lighting variations and artifacts.

In [ ]:
import cv2
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import glob

base_dir = '/content/data_odir'
csv_file_matches = glob.glob(base_dir + '/**/full_df.csv', recursive=True)
if not csv_file_matches:
    raise FileNotFoundError("full_df.csv not found in the extracted archive.")
csv_file = csv_file_matches[0]

df = pd.read_csv(csv_file)
print(f"Loaded full_df.csv with {len(df)} entries.")

output_dir = '/content/preprocessed_images'
os.makedirs(output_dir, exist_ok=True)

def crop_image_from_gray(img, tol=7):
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1),mask.any(0))]
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol
        check_shape = img[:,:,0][np.ix_(mask.any(1),mask.any(0))].shape[0]
        if (check_shape == 0):
            return img
        else:
            img1=img[:,:,0][np.ix_(mask.any(1),mask.any(0))]
            img2=img[:,:,1][np.ix_(mask.any(1),mask.any(0))]
            img3=img[:,:,2][np.ix_(mask.any(1),mask.any(0))]
            img = np.dstack([img1,img2,img3])
        return img

def apply_ben_graham(img, image_size=380):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = crop_image_from_gray(img)
    img = cv2.resize(img, (image_size, image_size))
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0,0), image_size/10), -4, 128)
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

def find_image(filename):
    for filepath in glob.iglob(base_dir + f"/**/{filename}", recursive=True):
        return filepath
    return None

success, fail = 0, 0
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing Images"):
    filename = row.get("filename")
    if pd.isna(filename): continue
    out_path = os.path.join(output_dir, filename)
    if os.path.exists(out_path):
        success += 1
        continue
        
    img_path = find_image(filename)
    if not img_path:
        fail += 1
        continue
        
    img = cv2.imread(img_path)
    if img is None:
        fail += 1
        continue
        
    try:
        res = apply_ben_graham(img, image_size=380)
        cv2.imwrite(out_path, res)
        success += 1
    except Exception as e:
        fail += 1

print(f"Preprocessing complete. Success: {success}, Fails: {fail}")

### 3. Model Training & ReduceLROnPlateau

In [ ]:
import ast
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score, classification_report
from skmultilearn.model_selection import iterative_train_test_split
import matplotlib.pyplot as plt
import datetime

model_dir = '/content/drive/MyDrive/Okul/ClicinicaEye-NLP/models/Vision'
os.makedirs(model_dir, exist_ok=True)
model_save_path = os.path.join(model_dir, 'odir_efficientnet_b4.pth')
report_save_path = os.path.join(model_dir, 'evaluation_report.md')

DISEASE_CLASSES = ["N", "D", "G", "C", "A", "H", "M", "O"]

class ODIRDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        if self.transform:
            image = self.transform(image)
        return image, label

# Build Dataset matching preprocessed valid images
valid_paths, valid_labels = [], []
for _, row in df.iterrows():
    fname = row.get('filename')
    if pd.isna(fname): continue
    path = os.path.join(output_dir, fname)
    if os.path.exists(path):
        valid_paths.append(path)
        target_list = ast.literal_eval(row['target'])
        valid_labels.append(target_list)
        
X = np.array(valid_paths).reshape(-1, 1)
y = np.array(valid_labels)

print(f"Loaded {len(X)} valid training images.")

# Multi-Label Stratified Split
X_train, y_train, X_val, y_val = iterative_train_test_split(X, y, test_size=0.2)
X_train, X_val = X_train.flatten(), X_val.flatten()

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ODIRDataset(X_train, y_train, transform=train_transform)
val_ds = ODIRDataset(X_val, y_val, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Accelerating using: {device}")

# Calculate Dynamic Pos_Weight
num_pos = y_train.sum(axis=0)
num_neg = len(y_train) - num_pos
pos_weight = torch.tensor(num_neg / (num_pos + 1e-6), dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, len(DISEASE_CLASSES))
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

### NOTE ON VERBOSE: In newer PyTorch versions, 'verbose' in ReduceLROnPlateau is deprecated.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

epochs = 20
best_f1 = 0.0
best_epoch = 0
history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
final_classification_report = ""

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    model.eval()
    val_loss = 0.0
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outputs = model(imgs)
            
            loss = criterion(outputs, lbls)
            val_loss += loss.item()
            
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(lbls.cpu().numpy())
            
    train_l = train_loss/len(train_loader)
    val_l = val_loss/len(val_loader)
    val_f1 = f1_score(np.array(all_targets), np.array(all_preds), average='macro', zero_division=0)
    
    history['train_loss'].append(train_l)
    history['val_loss'].append(val_l)
    history['val_f1'].append(val_f1)
    
    print(f"Epoch {epoch+1} | Train Loss: {train_l:.4f} | Val Loss: {val_l:.4f} | Val F1: {val_f1:.4f} | LR: {optimizer.param_groups[0]['lr']}")
    
    # Max mode scheduler based on Val F1
    scheduler.step(val_f1)
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch + 1
        final_classification_report = classification_report(np.array(all_targets), np.array(all_preds), target_names=DISEASE_CLASSES, zero_division=0)
        print(f"⭐ New Best Val F1: {best_f1:.4f}. Saving Model directly to Drive... ⭐")
        torch.save(model.state_dict(), model_save_path)

# 7. Generate Evaluation Report
report_content = f"""# ClinicaEye-NLP Vision Evaluation Report

**Date:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Model:** EfficientNet-B4 (Pre-trained on ImageNet)
**Task:** Multi-label Fundus Image Classification (ODIR-5K)

## Summary Metrics
- **Best Validation F1-Macro:** {best_f1:.4f}
- **Epoch achieved:** {best_epoch}
- **Final Training Loss:** {history['train_loss'][-1]:.4f}
- **Final Validation Loss:** {history['val_loss'][-1]:.4f}

## Detailed Classification Report
```
{final_classification_report}
```
"""
with open(report_save_path, 'w') as f:
    f.write(report_content)
print(f"Evaluation report saved to {report_save_path}")

### 4. Training Analytics Visualization

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(history['train_loss'], label='Train Loss', color='cyan', marker='o')
ax1.plot(history['val_loss'], label='Val Loss', color='magenta', marker='x')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss', color='white')
ax1.tick_params(axis='y', labelcolor='white')

ax2 = ax1.twinx()
ax2.plot(history['val_f1'], label='Val F1-Macro', color='lime', marker='s', linestyle='dashed')
ax2.set_ylabel('F1 Score', color='white')
ax2.tick_params(axis='y', labelcolor='white')

fig.legend(loc='center right', bbox_to_anchor=(0.85, 0.5))
plt.title('Colab Vision Output: Loss vs F1-Macro')
plt.grid(True, alpha=0.2)
plt.show()